# Module 4 — CPU solutions

No model download, GPU or paid service is needed. These are formative exercises with explicit fixtures. They do not count as measured capstone evidence. Run all cells in order; assertions check the worked results.

## 1. Distillation soft targets

Compute temperature-scaled KL(teacher || student), including T squared. A matched student should have zero loss.

In [ ]:
import math
def softmax(z,T):
    v=[math.exp((x-max(z))/T) for x in z];return [x/sum(v) for x in v]
def loss(teacher,student,T):
    p=softmax(teacher,T);q=softmax(student,T)
    return T*T*sum(a*math.log(a/b) for a,b in zip(p,q))
teacher=[5,3,1,-2]
assert abs(loss(teacher,teacher,2))<1e-12
assert abs(loss(teacher,[2,2,2,2],2)-1.8937977618733446)<1e-10
print(loss(teacher,[2,2,2,2],2))


## 2. Gate before speed

Use a maximum 5 percentage-point format drop and .05 raw cosine drop. Reject a faster candidate that fails either requirement.

In [ ]:
reference={"format":.9,"cosine":.8}
arms={"int8":{"format":.88,"cosine":.78,"seconds":2},"nf4":{"format":.8,"cosine":.79,"seconds":1}}
def passes(x):return 100*(reference["format"]-x["format"])<=5+1e-9 and reference["cosine"]-x["cosine"]<=.05+1e-9
assert passes(arms["int8"]) and not passes(arms["nf4"])
print({k:passes(v) for k,v in arms.items()})


## 3. Actual throughput

Count generated IDs, not requested maximum tokens. Use total new tokens divided by total measured seconds.

In [ ]:
runs=[{"new_ids":[1,2,3],"seconds":2.},{"new_ids":[4,5],"seconds":1.}]
throughput=sum(len(r["new_ids"]) for r in runs)/sum(r["seconds"] for r in runs)
assert abs(throughput-5/3)<1e-12
print({"tokens_per_second":throughput,"token_scope":"new IDs including EOS if present"})


## 4. Cost assumptions

Estimate sequential active GPU cost for 1,000 calls at 2 seconds/call and an explicitly illustrative $0.36/hour. Then account for 50% utilization; do not treat this as a current market quote.

In [ ]:
def cost(seconds,price,utilization):
    if not 0<utilization<=1 or min(seconds,price)<0:raise ValueError("Invalid assumptions")
    return seconds*price*1000/(3600*utilization)
assert abs(cost(2,.36,1)-.2)<1e-12 and abs(cost(2,.36,.5)-.4)<1e-12
print({"illustrative_active_cost":cost(2,.36,1),"at_half_utilization":cost(2,.36,.5)})


## Transfer to the model lab

Open `lab.ipynb`. Identify the corresponding real-data function, record the difference between fixture and model evidence, and explain one failure that a numerical score alone would miss.